# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR$^2$ dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library.

### Dataset Source
The Croissant schema and metadata are provided via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

All elements (record sets, fields, etc.) are referenced by their `@id` as specified in the Croissant schema.

In [ ]:
# Install mlcroissant if it's not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic summary
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets, their fields, and their `@id` values.

In [ ]:
# Retrieve all record sets and their @id
record_sets = [rs for rs in dataset.metadata.record_sets]
print(f"Available record sets (@id, name):\n")
for rs in record_sets:
    print(f"@id: {rs.id}", f"| Name: {rs.name}")

# For each record set, show its fields
for rs in record_sets:
    print(f"\nFields for record set @id={rs.id}:")
    for field in rs.fields:
        print(f"  - @id: {field.id} | Name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a selected record set (referenced by its `@id`) into a pandas DataFrame for further analysis.

_**Note**: You can adapt the list of `record_set_ids` below to extract additional record sets if needed._

In [ ]:
# Select record set(s) by @id based on cell above (typically only one main one for tabular datasets)
# Example main record set id (replace with the actual available @id)
main_record_set_id = dataset.metadata.record_sets[0].id  # Take first for demo, or set explicitly
record_set_ids = [main_record_set_id]

dataframes = {}

for record_set_id in record_set_ids:
    # Load all records of the record set
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

print(f"Loaded columns from record set {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing and transformation operations:
- Filter rows based on a numeric field,
- Normalize a numeric field,
- Group by a categorical field.

Refer to field `@id` and column names exactly as displayed in the extraction section above.

In [ ]:
# Identify a numeric field and a possible group field for EDA
# Replace these with actual column names/@ids from your loaded DataFrame if they differ
df = dataframes[main_record_set_id]
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Using numeric field for demonstration: {numeric_field}")
else:
    print("No numeric columns found in the record set!")
    numeric_field = None

# Try to find a likely group field (e.g. a string/categorical field)
categorical_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = None
for col in categorical_fields:
    if col != numeric_field:
        group_field = col
        break

if numeric_field:
    threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 10
    # Filter based on the threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field (if available)
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and compare groups if possible. Requires matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group field exists, plot distribution by group
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading and initial exploration of a mass spectrometry dataset using the `mlcroissant` library.

- The dataset schema is machine readable and exposes its structure via record set and field `@id`s.
- Data can be easily loaded and processed using Croissant and pandas workflows.
- Exploratory analysis and visualizations help in understanding the distributions and possible groupings in the biological data.

Continue to apply domain-specific filters and modeling as appropriate for your analysis!
